# Motif-motif interaction

In [ ]:
import pandas as pd
import os
import glob
import numpy as np

## Preprocessing-Integration

In [ ]:
import os
import glob
import pandas as pd

# --- 1. Set file path ---
directory_path = './tf/tf_tf/'

# Check if the directory exists
if not os.path.isdir(directory_path):
    print(f"Error: Directory '{directory_path}' does not exist. Please check the path.")
else:
    # --- 2. Find all .csv files ---
    csv_files = glob.glob(os.path.join(directory_path, '*.csv'))
    
    if not csv_files:
        print(f"Warning: No .csv files found in directory '{directory_path}'.")
    else:
        print(f"Found {len(csv_files)} .csv files, starting processing...")
        
        # --- 3. Prepare a list to store each processed DataFrame ---
        all_dataframes = []

        # --- 4. Iterate through and process each file ---
        for file_path in csv_files:
            try:
                # a. Extract the filename to serve as the final row index
                file_name_as_index = os.path.basename(file_path).removesuffix('.csv')
                
                # b. Read the CSV file
                # Use index_col=0 to treat the first column of the CSV as the initial index
                # This correctly reads your single-row data
                single_row_df = pd.read_csv(file_path, index_col=0)
                
                # c. Rename the row index to the filename
                # The row name might be '0' after reading; we replace it here
                single_row_df.index = [file_name_as_index]
                
                # d. Append the processed single-row DataFrame to the list
                all_dataframes.append(single_row_df)
                
                print(f"  - Processed: {os.path.basename(file_path)}")

            except Exception as e:
                print(f"  - Error occurred while processing file '{file_path}': {e}")

        # --- 5. Concatenate all DataFrames row-wise ---
        if all_dataframes:
            # pd.concat automatically takes the union of all columns (join='outer')
            # It aligns column names automatically; missing values are filled with NaN
            final_df = pd.concat(all_dataframes)
            
            # (Optional) Replace NaN values with 0
            final_df = final_df.fillna(0)
            
            # --- 6. Display and save results ---
            print("\n--- Merge Complete ---")
            print("Final generated DataFrame (first 5 rows):")
            print(final_df.head())
            
            # Save to a new CSV file
            output_path = 'tf_tf_combined.csv'
            final_df.to_csv(output_path)
            print(f"\nResults saved to: {output_path}")
        else:
            print("\nNo files were successfully processed; cannot generate final result.")

## Filter: remove Accessibility, take absolute value, merge bidirectional edges

In [ ]:
data = pd.read_csv('./tf/tf_tf/tf_tf_combined.csv', index_col=0)
data

In [ ]:
# 1. Find columns whose names contain 'Accessibility'
cols_to_drop = [col for col in data.columns if 'Accessibility' in col]
# Alternative Pandas implementation:
# cols_to_drop = data.columns[data.columns.str.contains('Accessibility')]

print("Columns to remove:")
print(cols_to_drop)

# 2. Drop these columns
# axis=1 selects columns
data_cleaned = data.drop(columns=cols_to_drop)

print("\nDataFrame after cleanup:")
print(data_cleaned)

In [ ]:
data_positive_abs = data_cleaned.abs()

In [ ]:
data_positive_abs

In [ ]:
# 1. Collect column names and create a set for fast lookup
all_columns = data_positive_abs.columns
column_set = set(all_columns)

# 2. Track canonical pairs to avoid double counting
found_pairs = set()
# Collect paired column names for inspection
reversed_columns_list = []

# 3. Iterate over all column names
for col_name in all_columns:
    # Split the column name on "->"
    parts = col_name.split('->')
    
    # Require exactly two parts separated by '->'
    if len(parts) == 2:
        part1, part2 = parts[0], parts[1]
        
        # Construct the reverse column name
        reverse_name = f"{part2}->{part1}"
        
        # Create an order-independent canonical key using frozenset
        # frozenset({'A', 'B'}) and frozenset({'B', 'A'}) are equivalent
        canonical_pair = frozenset((part1, part2))
        
        # Core logic:
        # 1. Check whether the reverse name exists
        # 2. Check whether the canonical pair has already been recorded
        if reverse_name in column_set and canonical_pair not in found_pairs:
            # Both conditions identify a new pair
            found_pairs.add(canonical_pair)
            reversed_columns_list.append((col_name, reverse_name))

# 4. Report the results
count = len(found_pairs)

print(f"Check complete.\nAmong {len(all_columns)} columns, found {count} pairs of reversed column names.")

if count > 0:
    print("\nPaired columns:")
    # Print only the first 10 pairs
    for i, pair in enumerate(reversed_columns_list[:10]):
        print(f"  {i+1}: {pair[0]} <---> {pair[1]}")
    if count > 10:
        print(f"  ... ({count - 10} additional pairs not shown)")

In [ ]:
# 1. Collect the new columns as Pandas Series
new_columns_list = []
# Track original columns that have already been processed
processed_columns = set()

# 2. Iterate over the original columns
for col_name in data_positive_abs.columns:
    if col_name in processed_columns:
        continue
    
    parts = col_name.split('->')
    if len(parts) != 2:
        # Retain columns that do not match the expected format
        new_columns_list.append(data_positive_abs[col_name])
        continue
        
    part1, part2 = parts[0], parts[1]
    reverse_name = f"{part2}->{part1}"
    canonical_name = "_".join(sorted((part1, part2)))
    
    # 3. Check whether the reverse column exists
    if reverse_name in data_positive_abs.columns:
        # --- Case 1: both directions exist ---
        # a. Compute the new column values
        new_col_data = data_positive_abs[col_name] + data_positive_abs[reverse_name]
        # b. Assign a canonical name to the new column
        new_col_data.name = canonical_name
        
        # c. Append the new Series to the list
        new_columns_list.append(new_col_data)
        
        # d. Mark both original columns as processed
        processed_columns.add(col_name)
        processed_columns.add(reverse_name)
    else:
        # --- Case 2: only one direction exists ---
        # a. Copy the values of the one-way interaction
        new_col_data = data_positive_abs[col_name]
        # b. Assign a canonical name to the new column
        new_col_data.name = canonical_name
        
        # c. Append the new Series to the list
        new_columns_list.append(new_col_data)
        
        # d. Mark the column as processed
        processed_columns.add(col_name)

# 4. Concatenate the collected columns into the final DataFrame
# axis=1 concatenates columns
undirected_df = pd.concat(new_columns_list, axis=1)

# 5. Optionally sort the final column names
undirected_df = undirected_df.reindex(sorted(undirected_df.columns), axis=1)

print("Undirected DataFrame assembled from the collected columns:")
print(undirected_df)

In [ ]:
47605-16613

In [ ]:
undirected_df.to_csv('./tf/tf_tf/processed.csv')

## Statistically specific and shared

In [ ]:
non_zero_counts_per_column = (undirected_df != 0).sum(axis=0)

In [ ]:
most_common_interactions = non_zero_counts_per_column.sort_values(ascending=False)
print("\nMost prevalent regulatory relationships (present in the most cell types):")
print(most_common_interactions.head())

In [ ]:
most_common_interactions.to_csv('./tf/tf_tf/all_30992.csv')

In [ ]:
meta = pd.DataFrame(most_common_interactions)
meta[0].value_counts()

### specific

In [ ]:
# --- Step 1: count cell types containing each interaction ---

# (undirected_df > 0) produces a Boolean matrix
# .sum(axis=0) counts True values in each column
non_zero_counts = (undirected_df > 0).sum(axis=0)

print("Number of cell types containing each interaction:")
print(non_zero_counts)
print("-" * 40)


# --- Step 2: select interactions present in exactly one cell type ---

specific_interaction_names = non_zero_counts[non_zero_counts == 1].index

print("Interaction pairs present in exactly one cell type:")
print(list(specific_interaction_names))
print("-" * 40)


# --- Step 3: extract details of cell-type-specific interactions ---

# 1. Retain only columns for cell-type-specific interactions
specific_df = undirected_df[specific_interaction_names]

# 2. Stack the DataFrame into a Series after masking zero values below
# Replace zeros with NaN so they are excluded by stack()
stacked_series = specific_df.replace(0, pd.NA).stack()

# 3. Reset the Series index to obtain the desired table
specific_interactions_details = stacked_series.reset_index()

# 4. Rename columns for clarity
specific_interactions_details.columns = ['CellType', 'Interaction', 'Strength']


# --- Print the final results ---
print("Details of cell-type-specific interactions:")
if specific_interactions_details.empty:
    print("No interactions present in exactly one cell type were found.")
else:
    print(specific_interactions_details)

In [ ]:
specific_interactions_details.to_csv('./tf/tf_tf/specific.csv')

In [ ]:
specific_interactions_details['CellType'].value_counts()

In [ ]:
strength_series = specific_interactions_details['Strength']

In [ ]:
plt.figure(figsize=(10, 6))

# Plot the distribution of 'Strength' using Seaborn
sns.histplot(strength_series, kde=True, bins=15, color='purple')

plt.title('Distribution of Interaction Strengths for Specific Pairs')
plt.xlabel('Interaction Strength')
plt.ylabel('Frequency (Count)')
plt.grid(True, linestyle='--', alpha=0.7)

# Save the image
# plt.savefig('specific_strength_distribution.png')
plt.show()

### common

In [ ]:
# --- Step 1: identify interactions present in all cell types ---

# 1. Get the number of cell types (DataFrame rows)
num_cell_types = len(undirected_df)
print(f"There are {num_cell_types} cell types.")

# 2. Count cell types containing each interaction (nonzero values per column)
non_zero_counts = (undirected_df > 0).sum(axis=0)

# 3. Select interactions whose occurrence count equals the number of cell types
common_interaction_names = non_zero_counts[non_zero_counts == num_cell_types].index

print(f"\nCommon interactions present in all cell types:")
print(list(common_interaction_names))
print("-" * 50)


# --- Step 2: sum the values for these common interactions ---

if not common_interaction_names.empty:
    # 1. Select common interaction columns from the original DataFrame
    common_df = undirected_df[common_interaction_names]

    # 2. Sum each selected column (axis=0)
    total_strength_of_common = common_df.sum(axis=0)

    # --- Print the final results ---
    print("Total interaction strength across cell types for common interactions:")
    print(total_strength_of_common)
else:
    print("No TF interactions present in all cell types were found.")

In [ ]:
pd.DataFrame(total_strength_of_common).to_csv('./common_1489.csv')

## Holistic exploration

In [ ]:
import scanpy as sc
import anndata as ad

In [ ]:
adata = ad.AnnData(undirected_df)
adata

In [ ]:
adata.obs['cell_type'] = adata.obs_names.tolist()
adata.obs['cell_type'] = adata.obs['cell_type'].astype('category')

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000)

In [ ]:
adata

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20)

In [ ]:
sc.tl.pca(adata)

In [ ]:
sc.tl.dendrogram(adata, "cell_type")

In [ ]:
sc.pl.dendrogram(adata, "cell_type")

## Motif cluster

In [ ]:
motif = pd.read_csv('./tf/motif_cluster.csv', index_col=0)
motif

In [ ]:
moti = pd.read_csv('./tf/motif_number.csv')
moti

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

count_series = moti['count']

plt.figure(figsize=(10, 6))

# Plot the distribution of 'count' using Seaborn
# kde=True also draws a smoothed density curve
sns.histplot(count_series, kde=True, bins=30, color='royalblue')

plt.title("Distribution of 'count' Column")
plt.xlabel("Count Value")
plt.ylabel("Frequency")
plt.grid(True, linestyle='--', alpha=0.7)

# Save the image
# plt.savefig('moti_count_distribution.png')
plt.show()

In [ ]:
282-194

# TF-TF interaction

In [ ]:
import pandas as pd

## motif-tf

In [ ]:
gene = pd.read_csv('./tf/features.csv', header=None)
gene

In [ ]:
gene_0_list = gene[0].tolist()

In [ ]:
tf = pd.read_csv('./tf/motif_tf.csv')
tf

In [ ]:
gene_list = tf['tf'].tolist()

In [ ]:
set_gene_list = set(gene_list)
set_gene_0 = set(gene_0_list)

if set_gene_list.issubset(set_gene_0):
    print("Result: yes. All genes in gene_list are present in gene[0].tolist().")
else:
    print("Result: no. Some genes in gene_list are absent from gene[0].tolist().")
    
    missing_genes = set_gene_list - set_gene_0
    print("\nMissing genes:")
    print(list(missing_genes))

In [ ]:
tf['motif'].value_counts()

In [ ]:
pd.DataFrame(tf['motif'].value_counts()).to_csv('./tf/tf_number.csv')

In [ ]:
282-145+1

In [ ]:
len(set_gene_list)

## tf_tf

In [ ]:
data = pd.read_csv('./tf/tf_tf/motif_motif/processed.csv', index_col=0)
data

In [ ]:
motif_motif = data.columns.tolist()

In [ ]:
import pandas as pd
from itertools import product

tf_df = tf

# Example 'motif_motif' list
motif_motif_list = motif_motif

# --- 2. Core Processing Logic (Final Version) ---

motif_to_tf_map = tf_df.groupby('motif')['tf'].apply(list).to_dict()
results = []

# Iterate through each original motif interaction
for interaction in motif_motif_list:
    parts = interaction.split('_')
    if len(parts) != 2:
        continue
    
    motif1, motif2 = parts
    
    tf_list1 = motif_to_tf_map.get(motif1, [])
    tf_list2 = motif_to_tf_map.get(motif2, [])
    
    if not tf_list1 or not tf_list2:
        continue
        
    # --- This is the key modification point ---
    # Create a temporary set to store deduplicated TF interactions for the current motif_motif
    local_tf_interactions = set()
    
    # Generate all possible pairwise combinations
    for tf1, tf2 in product(tf_list1, tf_list2):
        # Exclude self-interactions (TF interacting with itself)
        if tf1 != tf2:
            # Create canonical format (alphabetically sorted)
            canonical_interaction = '_'.join(sorted([tf1, tf2]))
            # Add the canonical TF interaction to the temporary set; the set handles deduplication automatically
            local_tf_interactions.add(canonical_interaction)
            
    # Add all deduplicated TF interactions corresponding to this motif_motif to the final results
    for tf_interaction in local_tf_interactions:
        results.append([interaction, tf_interaction])

# --- 3. Create the final DataFrame ---

tf_interaction_df = pd.DataFrame(results, columns=['motif_motif', 'tf_tf'])

# Print final results
print("Original 'tf' mapping DataFrame:")
print(tf_df)
print("\nFinal generated TF interaction information (deduplicated within each motif_motif):")
print(tf_interaction_df)

In [ ]:
tf_interaction_df

In [ ]:
tf_interaction_df.to_csv('./tf/tf_tf/motif_tf_all.csv')

In [ ]:
tf_interaction_df['motif_motif'].value_counts()

In [ ]:
tf_interaction_df['tf_tf'].value_counts()

In [ ]:
now = pd.DataFrame(tf_interaction_df['motif_motif'].value_counts())
now

In [ ]:
set(motif_motif) - set(now.index.tolist())

## common

In [ ]:
common = pd.read_csv('./tf/tf_tf/motif_motif/common_1489.csv')
common.columns = ['motif_motif', 'Strength']
common

In [ ]:
common_df = pd.merge(tf_interaction_df, common, on='motif_motif', how='inner')

In [ ]:
common_df

In [ ]:
com = pd.DataFrame(common_df['tf_tf'].value_counts())
com

## spe

In [ ]:
spe = pd.read_csv('./tf/tf_tf/motif_motif/specific_ 7156.csv', index_col=0)
spe

In [ ]:
spe.columns = ['CellType','motif_motif', 'Strength']

In [ ]:
spe_df = pd.merge(tf_interaction_df, spe, on='motif_motif', how='inner')

In [ ]:
spe_df

In [ ]:
spe = pd.DataFrame(spe_df['tf_tf'].value_counts())

In [ ]:
spe

## Remove redundancy between the two

In [ ]:
intsect = list(set(spe.index) & set(com.index))
len(intsect)

## merge commom

In [ ]:
aggregated_df = common_df.groupby('tf_tf').agg(
    Summed_Strength=('Strength', 'sum'),
    
    Motif_Sources=('motif_motif', lambda x: ', '.join(x))
).reset_index() 
aggregated_df

In [ ]:
common = aggregated_df[~aggregated_df['tf_tf'].isin(intsect)]
common

In [ ]:
common.columns = ['tf_tf', 'score', 'motif_motif']

In [ ]:
split_series = common_df['motif_motif'].str.split(', ')

exploded_series = split_series.explode()

unique_count = exploded_series.nunique()

In [ ]:
exploded_series

In [ ]:
unique_count

In [ ]:
common.to_csv('./tf/tf_tf/common_32374.csv')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

plt.figure(figsize=(10, 6))
sns.histplot(common['score'], kde=True, bins=5)

plt.show()

In [ ]:
sns.violinplot(y='score', data=common)

plt.tight_layout()
plt.show()

## specific

In [ ]:
spe_df

In [ ]:
spe_df = spe_df[~spe_df['tf_tf'].isin(intsect)]
spe_df

In [ ]:
spe = pd.DataFrame(spe_df['tf_tf'].value_counts())
spe

In [ ]:
spe['count'].value_counts()

In [ ]:
spe_1 = spe[spe['count'] == 1]
spe_1

In [ ]:
spe_df = spe_df[spe_df['tf_tf'].isin(spe_1.index.tolist())]
spe_df

In [ ]:
spe_df['motif_motif'].value_counts()

In [ ]:
spe_df.columns = ['motif_motif', 'tf_tf', 'cell_type', 'score']

In [ ]:
spe_df.to_csv('./tf/tf_tf/specific_36085.csv')

In [ ]:
spe_df['cell_type'].value_counts()

In [ ]:
plt.figure(figsize=(10, 7))

sns.violinplot(x='cell_type', y='score', data=spe_df)

# Add a title and axis labels

plt.xticks(rotation=90, fontsize=11)

# Display the figure
plt.tight_layout()
plt.show()

# Combine expression

In [ ]:
import pandas as pd

## common

In [ ]:
common = pd.read_csv('./tf/tf_tf/common_32374.csv', index_col=0)
common

In [ ]:
split_df = common['tf_tf'].str.split('_', expand=True)

stacked_series = split_df.stack()

unique_tf_count = stacked_series.nunique()
unique_tf_count

In [ ]:
unique_parts_count = common['motif_motif'].str.split(', ').explode().str.split('_').explode().nunique()
unique_parts_count

## specific

In [ ]:
specific = pd.read_csv('./tf/tf_tf/specific_36085.csv', index_col=0)
specific

In [ ]:
split_df = specific['tf_tf'].str.split('_', expand=True)

stacked_series = split_df.stack()

unique_tf_count = stacked_series.nunique()
unique_tf_count

In [ ]:
split_df = specific['motif_motif'].str.split('_', expand=True)

stacked_series = split_df.stack()

unique_tf_count = stacked_series.nunique()
unique_tf_count

## using pseudobulk

In [ ]:
import os
import pandas as pd

directory_path = './tf/rna/'

# --- 2. Core Code: Read, Process, and Concatenate Files ---

# Check if the directory exists
if not os.path.isdir(directory_path):
    print(f"Error: Directory '{directory_path}' does not exist.")
else:
    # Create an empty list to store data read from each file (as Series)
    all_data_series = []

    # Iterate through all files in the directory
    for filename in sorted(os.listdir(directory_path)):
        # Ensure we only process .csv files
        if filename.endswith('.csv'):
            # Construct the full file path
            full_path = os.path.join(directory_path, filename)
            
            # Extract column name: take the part before the first '.' in the filename
            column_name = filename.split('.')[0]
            
            # Read the CSV file as a Pandas Series
            # header=0 indicates the first row is the header
            # .iloc[:, 0] ensures we only take the first column of data
            # index_col=0 uses the first column of the file as the index
            data_series = pd.read_csv(full_path, header=0, index_col=0).iloc[:, 0]
            
            # Set the Series name attribute to the extracted column name
            data_series.name = column_name
            
            # Append this processed Series to the list
            all_data_series.append(data_series)

    # Check if any CSV files were found
    if not all_data_series:
        print("Error: No CSV files found in the specified directory.")
    else:
        # Use pd.concat to concatenate all Series in the list column-wise (axis=1)
        final_df = pd.concat(all_data_series, axis=1)
        
        # Print the final DataFrame
        print("Successfully merged DataFrame:")
        print(final_df)

In [ ]:
tf = pd.read_csv('./tf/motif_tf.csv')
tf

In [ ]:
tf_list = list(set(tf['tf']))
len(tf_list)

In [ ]:
tf_mat = final_df.loc[tf_list,:]
tf_mat

In [ ]:
tf_mat.to_csv('./tf/tf_tf/tf_specific_expression.csv')

## common，Comprehensive gene expression amount and expression range

In [ ]:
common = pd.DataFrame(tf_mat.sum(axis=1))
common.columns = ['sum']

In [ ]:
common['non_zero'] = (tf_mat != 0).sum(axis=1)
common

In [ ]:
common.to_csv('./tf/tf_tf/tf_common_expression.csv')

## specific，The ratio of expression levels in a specific cell type compared to expression levels in other cell types

In [ ]:
ratio_df = tf_mat.apply(
    lambda row: row / ((row.sum() - row) / (len(row) - 1)),
    axis=1
)
ratio_df

In [ ]:
ratio_df.to_csv('./tf/tf_tf/tf_specific_expression_ratio.csv')

## Filling tf_tf

### common

In [ ]:
com = pd.read_csv('./tf/tf_tf/common_32374.csv', index_col=0)
com

In [ ]:
com_exp = pd.read_csv('tf/tf_tf/tf_common_expression.csv')
com_exp

In [ ]:
import pandas as pd

# Step 1: Split the 'tf_tf' column in 'com' into 'tf1' and 'tf2'
com[['tf1', 'tf2']] = com['tf_tf'].str.split('_', expand=True)

# Step 2: First merge to populate information for 'tf1'
# Left merge 'com' and 'com_exp', matching com['tf1'] with com_exp['gene_name']
# The suffixes parameter distinguishes columns with the same name from both DataFrames (though not strictly needed here, it's good practice)
merged_df = pd.merge(
    com,
    com_exp,
    left_on='tf1',
    right_on='gene_name',
    how='left'
)

# Rename the columns populated after the first merge for distinction
merged_df = merged_df.rename(columns={'sum': 'tf1_sum', 'non_zero': 'tf1_non_zero'})
# Drop the redundant 'gene_name' column
merged_df = merged_df.drop(columns='gene_name')


# Step 3: Second merge to populate information for 'tf2'
# Perform another left merge on the result of the previous step, this time matching com['tf2'] with com_exp['gene_name']
final_df = pd.merge(
    merged_df,
    com_exp,
    left_on='tf2',
    right_on='gene_name',
    how='left'
)

# Rename the columns populated after the second merge
final_df = final_df.rename(columns={'sum': 'tf2_sum', 'non_zero': 'tf2_non_zero'})
# Drop the redundant 'gene_name' column again
final_df = final_df.drop(columns='gene_name')


# --- 3. Print Results ---

print("Original 'com' DataFrame:")
print(com.head())
print("\nOriginal 'com_exp' DataFrame:")
print(com_exp.head())
print("\n" + "="*50)
print("Final result populated with sum and non_zero information:")
# Reorder columns for better presentation
final_df = final_df[['tf_tf', 'tf1', 'tf2', 'score', 'motif_motif', 'tf1_sum', 'tf1_non_zero', 'tf2_sum', 'tf2_non_zero']]
print(final_df)

In [ ]:
final_df

In [ ]:
condition = (
    (final_df['score'] > 10) &
    (final_df['tf1_sum'] > 10) &
    (final_df['tf1_non_zero'] > 13) &
    (final_df['tf2_sum'] > 10) &
    (final_df['tf2_non_zero'] > 13)
)

filtered_df = final_df[condition]
filtered_df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

plt.figure(figsize=(10, 6))
sns.histplot(final_df['score'], kde=True, bins=5)

plt.show()

### specific

In [ ]:
spe = pd.read_csv('./tf/tf_tf/specific_36085.csv', index_col=0)
spe

In [ ]:
spe_exp = pd.read_csv('./tf/tf_tf/tf_specific_expression.csv', index_col=0)
spe_exp

In [ ]:
spe_ratio = pd.read_csv('./tf/tf_tf/tf_specific_expression_ratio.csv', index_col=0)
spe_ratio

In [ ]:
import pandas as pd
import numpy as np

# --- 2. Core Processing Logic ---

# Step 1: Split the 'tf_tf' column in 'spe' into 'tf1' and 'tf2'
spe[['tf1', 'tf2']] = spe['tf_tf'].str.split('_', expand=True)

# Step 2: Define a function to process a single row and perform lookups
def get_exp_ratio(row):
    # Get tf1, tf2, and cell_type from the current row
    tf1 = row['tf1']
    tf2 = row['tf2']
    cell_type = row['cell_type']
    
    # Use try-except to handle potential lookup failures (e.g., if a specific tf or cell_type does not exist)
    try:
        exp1 = spe_exp.loc[tf1, cell_type]
    except KeyError:
        exp1 = np.nan # Fill with NaN if lookup fails
        
    try:
        exp2 = spe_exp.loc[tf2, cell_type]
    except KeyError:
        exp2 = np.nan
        
    try:
        ratio1 = spe_ratio.loc[tf1, cell_type]
    except KeyError:
        ratio1 = np.nan
        
    try:
        ratio2 = spe_ratio.loc[tf2, cell_type]
    except KeyError:
        ratio2 = np.nan
        
    # Return a Series containing four new values
    return pd.Series([exp1, exp2, ratio1, ratio2], index=['tf1_exp', 'tf2_exp', 'tf1_ratio', 'tf2_ratio'])

# Step 3: Apply this function to each row of the spe DataFrame (axis=1)
new_columns = spe.apply(get_exp_ratio, axis=1)

# Step 4: Concatenate the newly generated columns back to the original spe DataFrame
final_spe_df = pd.concat([spe, new_columns], axis=1)

In [ ]:
final_spe_df

In [ ]:
condition = (
    (final_spe_df['score'] > 0.01) &
    (final_spe_df['tf1_exp'] > 0.5) &
    (final_spe_df['tf2_exp'] > 0.5) &
    (final_spe_df['tf1_ratio'] > 1) &
    (final_spe_df['tf2_ratio'] > 1)
)

filtered_spe_df = final_spe_df[condition]
filtered_spe_df

In [ ]:
filtered_spe_df['cell_type'].value_counts()   # monocyte

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 

sns.set_style("whitegrid")

plt.figure(figsize=(10, 6))
sns.histplot(final_spe_df['tf1_exp'], kde=True, bins=5)

plt.show()

## PPI visual

### common

In [ ]:
# filtered_df.to_csv('./tf/tf_tf/common_filtered_339.csv')
filtered_df = pd.read_csv('./tf/tf_tf/common_filtered_339.csv', index_col=0)
filtered_df

In [ ]:
# Get the unique TFs from 'tf1' and 'tf2' columns
unique_tfs = pd.unique(filtered_df[['tf1', 'tf2']].values.ravel('K'))

# Count the number of unique TFs
num_unique_tfs = len(unique_tfs)
num_unique_tfs

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# Set font and PDF output parameters
plt.rcParams['pdf.fonttype'] = 42      # Ensure text is embedded as TrueType fonts
plt.rcParams['ps.fonttype'] = 42       # If PostScript output is required

# Construct the graph
G = nx.from_pandas_edgelist(filtered_df, "tf1", "tf2", edge_attr="score")

# Calculate node centrality
centrality = nx.degree_centrality(G)

# Normalize node sizes (reduce range to avoid overlap)
node_sizes = np.interp(
    list(centrality.values()),
    (min(centrality.values()), max(centrality.values())),
    (100, 800)
)

# Edge color mapping
scores = [d['score'] for (_, _, d) in G.edges(data=True)]
norm = plt.Normalize(vmin=min(scores), vmax=max(scores))
edge_colors = cm.plasma(norm(scores))

# Improve layout: increase k value or use kamada_kawai
pos = nx.spring_layout(G, k=2.0, seed=42)
# pos = nx.kamada_kawai_layout(G)  # Alternative layout option

fig, ax = plt.subplots(figsize=(12, 10))

# Draw nodes
nx.draw_networkx_nodes(
    G, pos,
    node_size=node_sizes,
    node_color="skyblue",
    alpha=0.9,
    edgecolors="black",
    ax=ax
)

# Draw edges
nx.draw_networkx_edges(
    G, pos,
    edge_color=edge_colors,
    width=np.interp(scores, (min(scores), max(scores)), (0.5, 2)),
    alpha=0.3,
    ax=ax
)

# Labels (Show top 50, add offset to avoid overlapping nodes)
top_nodes = sorted(centrality, key=centrality.get, reverse=True)[:50]
for n in top_nodes:
    x, y = pos[n]
    ax.text(
        x, y+0.05,  # Shift upwards
        n,
        fontsize=9,
        ha="center", va="center",
        bbox=dict(facecolor="white", alpha=0.6, edgecolor="none", pad=1)  # White background to prevent overlap with edges
    )

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=cm.plasma, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label("Edge Score")

ax.set_title("Improved PPI Network (Reduced Overlap)", fontsize=14)
ax.axis("off")
plt.savefig("ppi_network.pdf", format="pdf", bbox_inches="tight")

plt.show()

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# Construct the graph
G = nx.from_pandas_edgelist(filtered_df, "tf1", "tf2", edge_attr="score")

# Calculate node centrality
centrality = nx.degree_centrality(G)

# Normalize node sizes
min_centrality = min(centrality.values())
max_centrality = max(centrality.values())
node_sizes = np.interp(
    list(centrality.values()),
    (min_centrality, max_centrality),
    (100, 800)
)

# Edge color mapping
scores = [d['score'] for (_, _, d) in G.edges(data=True)]
norm = plt.Normalize(vmin=min(scores), vmax=max(scores))
edge_colors = cm.plasma(norm(scores))

# Layout
pos = nx.spring_layout(G, k=2.0, seed=42)

fig, ax = plt.subplots(figsize=(14, 12))

# Draw nodes
nx.draw_networkx_nodes(
    G, pos,
    node_size=node_sizes,
    node_color="skyblue",
    alpha=0.9,
    edgecolors="black",
    ax=ax
)

# Draw edges
nx.draw_networkx_edges(
    G, pos,
    edge_color=edge_colors,
    width=np.interp(scores, (min(scores), max(scores)), (0.5, 2)),
    alpha=0.3,
    ax=ax
)

# Labels (Show top 20 only, with slight offset)
top_nodes = sorted(centrality, key=centrality.get, reverse=True)[:20]
for n in top_nodes:
    x, y = pos[n]
    ax.text(
        x, y+0.05, n,
        fontsize=9,
        ha="center", va="center",
        bbox=dict(facecolor="white", alpha=0.6, edgecolor="none", pad=1)
    )

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=cm.plasma, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.8)
cbar.set_label("Edge Score", fontsize=12)

# ==================== Code for adding node size legend ====================
# Create proxy artists for the legend
legend_handles = []
legend_labels = []
# Define representative centrality values to display in the legend
centrality_levels = np.linspace(min_centrality, max_centrality, num=4)
# Calculate node sizes corresponding to these centrality values
size_levels = np.interp(centrality_levels, (min_centrality, max_centrality), (100, 800))

for i, size in enumerate(size_levels):
    legend_handles.append(plt.scatter([], [], s=size, color='skyblue', edgecolor='black', alpha=0.9))
    legend_labels.append(f'{centrality_levels[i]:.2f}')

legend = ax.legend(legend_handles, legend_labels, title='Node Centrality', scatterpoints=1, labelspacing=1.5, frameon=True, loc='lower right')
legend.get_title().set_fontsize('12')
# =================================================================

ax.set_title("Improved PPI Network (Reduced Overlap)", fontsize=16)
ax.axis("off")

# Save as PDF (vector graphic, suitable for papers) and PNG
plt.savefig("ppi_network_with_legend.pdf", format="pdf", bbox_inches="tight")
plt.savefig("ppi_network_with_legend.png", format="png", bbox_inches="tight", dpi=300)

plt.show()

### spe

In [ ]:
# filtered_spe_df.to_csv('./tf/tf_tf/specific_filtered_1888.csv')
filtered_spe_df = pd.read_csv('./tf/tf_tf/specific_filtered_1888.csv', index_col=0)
filtered_spe_df

In [ ]:
filtered_spe_df['cell_type'].value_counts()

In [ ]:
filtered_spe_df = filtered_spe_df[filtered_spe_df['cell_type'] == 'dc']
filtered_spe_df

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# Construct the graph
G = nx.from_pandas_edgelist(filtered_spe_df, "tf1", "tf2", edge_attr="score")

# Calculate node centrality
centrality = nx.degree_centrality(G)

# Normalize node sizes
min_centrality = min(centrality.values())
max_centrality = max(centrality.values())
node_sizes = np.interp(
    list(centrality.values()),
    (min_centrality, max_centrality),
    (100, 800)
)

# Edge color mapping
scores = [d['score'] for (_, _, d) in G.edges(data=True)]
norm = plt.Normalize(vmin=min(scores), vmax=max(scores))
edge_colors = cm.plasma(norm(scores))

# Layout configuration
pos = nx.spring_layout(G, k=2.0, seed=42)

fig, ax = plt.subplots(figsize=(14, 12))

# Draw nodes
nx.draw_networkx_nodes(
    G, pos,
    node_size=node_sizes,
    node_color="skyblue",
    alpha=0.9,
    edgecolors="black",
    ax=ax
)

# Draw edges
nx.draw_networkx_edges(
    G, pos,
    edge_color=edge_colors,
    width=np.interp(scores, (min(scores), max(scores)), (0.5, 2)),
    alpha=0.3,
    ax=ax
)

# Labels (Show top 20 only, with slight offset)
top_nodes = sorted(centrality, key=centrality.get, reverse=True)[:20]
for n in top_nodes:
    x, y = pos[n]
    ax.text(
        x, y+0.05, n,
        fontsize=9,
        ha="center", va="center",
        bbox=dict(facecolor="white", alpha=0.6, edgecolor="none", pad=1)
    )

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=cm.plasma, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.8)
cbar.set_label("Edge Score", fontsize=12)

# ==================== Code for adding node size legend ====================
# Create proxy artists for the legend
legend_handles = []
legend_labels = []
# Define representative centrality values to display in the legend
centrality_levels = np.linspace(min_centrality, max_centrality, num=4)
# Calculate node sizes corresponding to these centrality values
size_levels = np.interp(centrality_levels, (min_centrality, max_centrality), (100, 800))

for i, size in enumerate(size_levels):
    legend_handles.append(plt.scatter([], [], s=size, color='skyblue', edgecolor='black', alpha=0.9))
    legend_labels.append(f'{centrality_levels[i]:.2f}')

legend = ax.legend(legend_handles, legend_labels, title='Node Centrality', scatterpoints=1, labelspacing=1.5, frameon=True, loc='lower right')
legend.get_title().set_fontsize('12')
# =================================================================

ax.set_title("Improved PPI Network (Reduced Overlap)", fontsize=16)
ax.axis("off")

# Save as PDF (vector graphic, suitable for papers) and PNG
plt.savefig("ppi_network_with_legend.pdf", format="pdf", bbox_inches="tight")
plt.savefig("ppi_network_with_legend.png", format="png", bbox_inches="tight", dpi=300)

plt.show()

## common validate alternatives

In [ ]:
com = pd.read_csv('./tf/tf_tf/common_32374.csv', index_col=0)
com

In [ ]:
com_exp = pd.read_csv('tf/tf_tf/tf_common_expression.csv')
com_exp

In [ ]:
com_exp[com_exp['gene_name'] == 'RORB']

In [ ]:
import pandas as pd

# Step 1: Split the 'tf_tf' column in 'com' into 'tf1' and 'tf2'
com[['tf1', 'tf2']] = com['tf_tf'].str.split('_', expand=True)

# Step 2: First merge to populate information for 'tf1'
# Left merge 'com' and 'com_exp', matching com['tf1'] with com_exp['gene_name']
# The suffixes parameter distinguishes columns with the same name from both DataFrames (though not strictly needed here, it's good practice)
merged_df = pd.merge(
    com,
    com_exp,
    left_on='tf1',
    right_on='gene_name',
    how='left'
)

# Rename the columns populated after the first merge for distinction
merged_df = merged_df.rename(columns={'sum': 'tf1_sum', 'non_zero': 'tf1_non_zero'})
# Drop the redundant 'gene_name' column
merged_df = merged_df.drop(columns='gene_name')


# Step 3: Second merge to populate information for 'tf2'
# Perform another left merge on the result of the previous step, this time matching com['tf2'] with com_exp['gene_name']
final_df = pd.merge(
    merged_df,
    com_exp,
    left_on='tf2',
    right_on='gene_name',
    how='left'
)

# Rename the columns populated after the second merge
final_df = final_df.rename(columns={'sum': 'tf2_sum', 'non_zero': 'tf2_non_zero'})
# Drop the redundant 'gene_name' column again
final_df = final_df.drop(columns='gene_name')


# --- 3. Print Results ---

print("Original 'com' DataFrame:")
print(com.head())
print("\nOriginal 'com_exp' DataFrame:")
print(com_exp.head())
print("\n" + "="*50)
print("Final result populated with sum and non_zero information:")
# Reorder columns for better presentation
final_df = final_df[['tf_tf', 'tf1', 'tf2', 'score', 'motif_motif', 'tf1_sum', 'tf1_non_zero', 'tf2_sum', 'tf2_non_zero']]
print(final_df)

In [ ]:
final_df

In [ ]:
r1 = final_df[final_df['tf1'] == 'RUNX1']
r1

In [ ]:
r2 = final_df[final_df['tf2'] == 'RUNX1']
r2

In [ ]:
ru = pd.concat([r1, r2])
ru

In [ ]:
condition = (
    (ru['score'] > 5) &
    (ru['tf1_sum'] > 8) &
    (ru['tf1_non_zero'] > 13) &
    (ru['tf2_sum'] > 8) &
    (ru['tf2_non_zero'] > 13)
)

filtered_df = ru[condition]
filtered_df

## spe validate alternatives

In [ ]:
spe = pd.read_csv('./tf/tf_tf/specific_filtered_1888.csv', index_col=0)
spe

In [ ]:
s1 = spe[spe['tf1'] == 'RUNX1']
s1

In [ ]:
s2 = spe[spe['tf2'] == 'RUNX1']
s2